# AI HealthOS - AI Microservice for Google Colab

This notebook runs the complete AI HealthOS AI Microservice on Google Colab with public URL access via ngrok.

## Features:
- **Symptom Checker**: NLP-based symptom analysis
- **Image Diagnosis**: CNN-based medical image analysis
- **OCR Service**: Text extraction from medical reports
- **Risk Prediction**: ML-based health risk assessment

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install fastapi uvicorn python-multipart pydantic pydantic-settings numpy pillow scikit-learn pandas python-jose passlib python-dotenv httpx pyngrok -q

# Install Tesseract OCR (optional - for real OCR)
!apt-get install -y tesseract-ocr -qq 2>/dev/null || echo "Tesseract installation skipped"

print("✅ All dependencies installed!")

## Step 2: Set Up Ngrok (for public URL)

In [ ]:
# Set your ngrok authtoken (get it from https://ngrok.com/)
# You can also set it as an environment variable

from pyngrok import ngrok
import os

# Option 1: Set your ngrok token directly (replace 'your-ngrok-auth-token-here' with your actual token)
NGROK_AUTH_TOKEN = "3C96iglKEoDEhz69CKuize5OsxP_dqHnSrKKEMSYGTxCxUJV"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Option 2: Use environment variable (uncomment and set NGROK_AUTH_TOKEN in your environment)
# NGROK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')

if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "your-ngrok-auth-token-here":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ Ngrok authenticated!")
else:
    print("⚠️ No ngrok token set. You can still run locally, but public URL won't work.")
    print("Get your free token at: https://ngrok.com/")


## Step 3: Define All Application Code

This cell contains all the code (schemas, routers, main app) in one place.

In [ ]:
# ============================================================================
# AI HealthOS - Complete AI Microservice Code
# ============================================================================

from fastapi import FastAPI, HTTPException, Depends, File, UploadFile, Header, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from pydantic_settings import BaseSettings
from functools import lru_cache
from contextlib import asynccontextmanager
from PIL import Image
from typing import List, Optional
from enum import Enum
import logging
import io
import os
import re
import random

# ============================================================================
# CONFIGURATION
# ============================================================================

class Settings(BaseSettings):
    """Application settings"""
    API_KEY: str = "ai-service-secret-key"
    HOST: str = "0.0.0.0"
    PORT: int = 8000
    DEBUG: bool = True
    MODEL_CACHE_DIR: str = "./model_cache"
    ENABLE_MOCK_MODELS: bool = True
    TESSERACT_CMD: str = "tesseract"
    MAX_IMAGE_SIZE: int = 10 * 1024 * 1024
    SUPPORTED_IMAGE_TYPES: list = ["image/jpeg", "image/png", "image/jpg"]
    
    class Config:
        env_file = ".env"
        case_sensitive = True

@lru_cache()
def get_settings() -> Settings:
    return Settings()

settings = get_settings()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================================================
# SCHEMAS / MODELS
# ============================================================================

class RiskLevel(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"
    CRITICAL = "CRITICAL"

class SymptomCheckRequest(BaseModel):
    symptoms: str = Field(..., description="Description of symptoms")
    age: Optional[int] = Field(None, description="Patient age")
    gender: Optional[str] = Field(None, description="Patient gender")
    additionalNotes: Optional[str] = Field(None, description="Additional notes")

class SymptomCheckResponse(BaseModel):
    predictedCondition: str
    riskLevel: str
    confidenceScore: float = Field(..., ge=0.0, le=1.0)
    recommendation: str
    possibleConditions: List[str]
    analysis: Optional[str] = None
    success: bool = True
    errorMessage: Optional[str] = None

class ImageDiagnosisResponse(BaseModel):
    predictedCondition: str
    confidenceScore: float = Field(..., ge=0.0, le=1.0)
    possibleConditions: List[str]
    recommendation: str
    analysis: Optional[str] = None
    success: bool = True
    errorMessage: Optional[str] = None

class OCRResponse(BaseModel):
    extractedText: str
    confidence: float = Field(..., ge=0.0, le=1.0)
    analysis: Optional[str] = None
    success: bool = True
    errorMessage: Optional[str] = None

class RiskPredictionRequest(BaseModel):
    age: int
    gender: str
    systolic: int
    diastolic: int
    bloodSugar: float
    heartRate: int

class RiskPredictionResponse(BaseModel):
    riskScore: float = Field(..., ge=0.0, le=1.0)
    riskCategory: str
    factors: List[str]
    recommendation: str
    success: bool = True
    errorMessage: Optional[str] = None

# ============================================================================
# SYMPTOM CHECKER LOGIC
# ============================================================================

CONDITIONS_DB = {
    "fever": {
        "conditions": ["Viral Fever", "Influenza", "COVID-19", "Malaria", "Typhoid"],
        "risk": "MEDIUM",
        "recommendation": "Rest, stay hydrated, and monitor temperature. Consult a doctor if fever persists beyond 3 days."
    },
    "headache": {
        "conditions": ["Tension Headache", "Migraine", "Sinusitis", "Cluster Headache"],
        "risk": "LOW",
        "recommendation": "Rest in a quiet, dark room. Stay hydrated. Consult a doctor if severe or persistent."
    },
    "chest pain": {
        "conditions": ["Angina", "Heart Attack", "Muscle Strain", "Acid Reflux", "Anxiety"],
        "risk": "CRITICAL",
        "recommendation": "SEEK IMMEDIATE MEDICAL ATTENTION. Call emergency services if severe."
    },
    "cough": {
        "conditions": ["Common Cold", "Bronchitis", "Pneumonia", "Allergies", "COVID-19"],
        "risk": "MEDIUM",
        "recommendation": "Stay hydrated, rest, and monitor. Consult a doctor if cough persists or breathing difficulty occurs."
    },
    "stomach pain": {
        "conditions": ["Gastritis", "Appendicitis", "Food Poisoning", "IBS", "Ulcer"],
        "risk": "MEDIUM",
        "recommendation": "Avoid spicy foods, stay hydrated. Seek immediate care if pain is severe or localized."
    },
    "skin rash": {
        "conditions": ["Allergic Reaction", "Eczema", "Psoriasis", "Contact Dermatitis", "Infection"],
        "risk": "MEDIUM",
        "recommendation": "Avoid scratching, keep area clean. Consult a dermatologist if persists or spreads."
    },
    "shortness of breath": {
        "conditions": ["Asthma", "Pneumonia", "Heart Failure", "Anxiety", "COPD"],
        "risk": "HIGH",
        "recommendation": "Seek medical attention immediately, especially if sudden or severe."
    },
    "diabetes": {
        "conditions": ["Type 2 Diabetes", "Type 1 Diabetes", "Pre-diabetes", "Gestational Diabetes"],
        "risk": "HIGH",
        "recommendation": "Monitor blood sugar regularly. Consult an endocrinologist for proper management."
    },
    "hypertension": {
        "conditions": ["Primary Hypertension", "Secondary Hypertension", "White Coat Hypertension"],
        "risk": "HIGH",
        "recommendation": "Monitor BP regularly, reduce salt intake, exercise. Consult a cardiologist."
    }
}

def analyze_symptoms_nlp(symptoms: str) -> dict:
    symptoms_lower = symptoms.lower()
    matched_condition = None
    for keyword, data in CONDITIONS_DB.items():
        if keyword in symptoms_lower:
            matched_condition = data.copy()
            matched_condition["matched_keyword"] = keyword
            break
    
    if not matched_condition:
        return {
            "conditions": ["General Malaise", "Viral Infection", "Stress-related Condition"],
            "risk": "LOW",
            "recommendation": "Monitor symptoms, rest, and stay hydrated. Consult a doctor if symptoms worsen.",
            "matched_keyword": "general"
        }
    return matched_condition

def calculate_confidence(symptoms: str, matched_keyword: str) -> float:
    base_confidence = 0.7
    if len(symptoms) > 50:
        base_confidence += 0.1
    if len(symptoms) > 100:
        base_confidence += 0.1
    confidence = min(0.95, base_confidence + random.uniform(-0.05, 0.05))
    return round(confidence, 2)

# ============================================================================
# IMAGE DIAGNOSIS LOGIC
# ============================================================================

SKIN_CONDITIONS = {
    "melanoma": {
        "name": "Melanoma",
        "description": "A serious form of skin cancer",
        "risk": "HIGH",
        "recommendation": "URGENT: Consult a dermatologist immediately for biopsy and treatment."
    },
    "basal_cell_carcinoma": {
        "name": "Basal Cell Carcinoma",
        "description": "Most common type of skin cancer",
        "risk": "MEDIUM",
        "recommendation": "Consult a dermatologist for treatment options."
    },
    "eczema": {
        "name": "Eczema (Dermatitis)",
        "description": "Inflammatory skin condition",
        "risk": "LOW",
        "recommendation": "Use moisturizers, avoid irritants. Consult dermatologist if severe."
    },
    "psoriasis": {
        "name": "Psoriasis",
        "description": "Autoimmune skin condition",
        "risk": "MEDIUM",
        "recommendation": "Consult a dermatologist for treatment plan."
    },
    "acne": {
        "name": "Acne",
        "description": "Common skin condition",
        "risk": "LOW",
        "recommendation": "Use gentle cleansers, avoid picking. Consult dermatologist if persistent."
    },
    "normal": {
        "name": "Normal Skin",
        "description": "No significant abnormalities detected",
        "risk": "LOW",
        "recommendation": "Continue regular skin care routine and monitor for changes."
    }
}

def preprocess_image(image_bytes: bytes) -> Image.Image:
    try:
        image = Image.open(io.BytesIO(image_bytes))
        if image.mode != 'RGB':
            image = image.convert('RGB')
        return image
    except Exception as e:
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Invalid image file")

def analyze_image_cnn(image: Image.Image) -> dict:
    conditions = list(SKIN_CONDITIONS.keys())
    weights = [0.1, 0.1, 0.2, 0.15, 0.2, 0.25]
    selected_key = random.choices(conditions, weights=weights)[0]
    condition = SKIN_CONDITIONS[selected_key]
    confidence = random.uniform(0.65, 0.95)
    alternatives = [SKIN_CONDITIONS[k]["name"] for k in conditions if k != selected_key][:3]
    return {
        "primary_condition": condition,
        "confidence": round(confidence, 2),
        "alternatives": alternatives
    }

# ============================================================================
# OCR LOGIC
# ============================================================================

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False

def mock_ocr_extraction(filename: str) -> tuple:
    mock_text = f"""
MEDICAL LABORATORY REPORT
Patient ID: {filename[:8] if len(filename) >= 8 else 'PATIENT1'}
Date: 2024-01-15

COMPLETE BLOOD COUNT (CBC)
-------------------------
Hemoglobin: 13.5 g/dL (Normal: 12-16)
White Blood Cells: 7,500 /cmm (Normal: 4,500-11,000)
Platelets: 250,000 /cmm (Normal: 150,000-400,000)
Red Blood Cells: 4.8 million/cmm (Normal: 4.2-5.4)

BLOOD CHEMISTRY
--------------
Glucose (Fasting): 95 mg/dL (Normal: 70-100)
Total Cholesterol: 195 mg/dL (Normal: <200)
LDL Cholesterol: 115 mg/dL (Normal: <130)
HDL Cholesterol: 45 mg/dL (Normal: >40)
Triglycerides: 140 mg/dL (Normal: <150)

LIVER FUNCTION TESTS
-------------------
SGOT (AST): 28 U/L (Normal: 5-40)
SGPT (ALT): 32 U/L (Normal: 7-56)
Alkaline Phosphatase: 75 U/L (Normal: 44-147)
Bilirubin: 0.8 mg/dL (Normal: 0.1-1.2)

REMARKS: All values within normal range.
    """
    return mock_text.strip(), 0.85

def analyze_medical_text(text: str) -> str:
    analysis_parts = []
    if re.search(r'glucose.*\d+', text.lower()):
        analysis_parts.append("Blood glucose levels mentioned")
    if re.search(r'cholesterol.*\d+', text.lower()):
        analysis_parts.append("Cholesterol levels mentioned")
    if re.search(r'hemoglobin.*\d+', text.lower()):
        analysis_parts.append("Hemoglobin levels mentioned")
    if re.search(r'normal', text.lower()):
        analysis_parts.append("Report indicates normal values")
    if re.search(r'high|elevated|abnormal', text.lower()):
        analysis_parts.append("Some values may be outside normal range")
    if not analysis_parts:
        analysis_parts.append("Medical report text extracted successfully")
    return "; ".join(analysis_parts)

# ============================================================================
# RISK PREDICTION LOGIC
# ============================================================================

def calculate_bp_risk(systolic: int, diastolic: int) -> tuple:
    if systolic >= 180 or diastolic >= 120:
        return 0.9, "Hypertensive Crisis"
    elif systolic >= 140 or diastolic >= 90:
        return 0.7, "Stage 2 Hypertension"
    elif systolic >= 130 or diastolic >= 80:
        return 0.5, "Stage 1 Hypertension"
    elif systolic >= 120:
        return 0.3, "Elevated"
    else:
        return 0.1, "Normal"

def calculate_glucose_risk(blood_sugar: float) -> tuple:
    if blood_sugar >= 200:
        return 0.8, "Very High"
    elif blood_sugar >= 140:
        return 0.6, "High"
    elif blood_sugar >= 100:
        return 0.4, "Pre-diabetic Range"
    else:
        return 0.1, "Normal"

def calculate_heart_rate_risk(heart_rate: int) -> tuple:
    if heart_rate > 120 or heart_rate < 50:
        return 0.8, "Abnormal"
    elif heart_rate > 100 or heart_rate < 60:
        return 0.4, "Borderline"
    else:
        return 0.1, "Normal"

def calculate_age_risk(age: int) -> float:
    if age >= 65:
        return 0.3
    elif age >= 45:
        return 0.2
    elif age >= 30:
        return 0.1
    else:
        return 0.05

def predict_health_risk_ml(request: RiskPredictionRequest) -> dict:
    factors = []
    risk_scores = []
    
    bp_risk, bp_category = calculate_bp_risk(request.systolic, request.diastolic)
    risk_scores.append(bp_risk)
    if bp_risk > 0.3:
        factors.append(f"Blood Pressure: {bp_category}")
    
    glucose_risk, glucose_category = calculate_glucose_risk(request.bloodSugar)
    risk_scores.append(glucose_risk)
    if glucose_risk > 0.3:
        factors.append(f"Blood Sugar: {glucose_category}")
    
    hr_risk, hr_category = calculate_heart_rate_risk(request.heartRate)
    risk_scores.append(hr_risk)
    if hr_risk > 0.3:
        factors.append(f"Heart Rate: {hr_category}")
    
    age_risk = calculate_age_risk(request.age)
    risk_scores.append(age_risk)
    if request.age > 60:
        factors.append(f"Age: {request.age} years (Senior)")
    
    weights = [0.35, 0.30, 0.25, 0.10]
    overall_risk = sum(score * weight for score, weight in zip(risk_scores, weights))
    
    if overall_risk >= 0.7:
        risk_category = "HIGH"
        recommendation = "Consult a healthcare provider immediately. Multiple risk factors detected."
    elif overall_risk >= 0.4:
        risk_category = "MEDIUM"
        recommendation = "Schedule a check-up with your doctor. Monitor your vitals regularly."
    elif overall_risk >= 0.2:
        risk_category = "LOW-MEDIUM"
        recommendation = "Maintain healthy lifestyle. Regular monitoring recommended."
    else:
        risk_category = "LOW"
        recommendation = "Continue healthy habits. Regular check-ups as per age guidelines."
    
    return {
        "risk_score": round(overall_risk, 2),
        "risk_category": risk_category,
        "factors": factors if factors else ["All vitals within normal range"],
        "recommendation": recommendation
    }

# ============================================================================
# CREATE FASTAPI APP
# ============================================================================

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("AI HealthOS Service starting up...")
    yield
    logger.info("AI HealthOS Service shutting down...")

app = FastAPI(
    title="AI HealthOS - AI Microservice",
    description="AI-powered medical diagnostic service with NLP, CNN, and ML models",
    version="1.0.0",
    lifespan=lifespan
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key != settings.API_KEY:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid API key")
    return x_api_key

# ============================================================================
# API ROUTES
# ============================================================================

@app.get("/")
async def root():
    return {"service": "AI HealthOS AI Microservice", "version": "1.0.0", "status": "running"}

@app.get("/health")
async def health_check():
    return {"status": "healthy", "service": "ai-healthos-ai-service"}

# --- Symptom Checker Routes ---
@app.post("/ai/symptom-check", response_model=SymptomCheckResponse)
async def check_symptoms(request: SymptomCheckRequest, api_key: str = Depends(verify_api_key)):
    try:
        logger.info(f"Analyzing symptoms: {request.symptoms[:50]}...")
        analysis = analyze_symptoms_nlp(request.symptoms)
        confidence = calculate_confidence(request.symptoms, analysis.get("matched_keyword", ""))
        risk_level = analysis["risk"]
        if request.age and request.age > 60 and risk_level == "LOW":
            risk_level = "MEDIUM"
        
        return SymptomCheckResponse(
            predictedCondition=analysis["conditions"][0],
            riskLevel=risk_level,
            confidenceScore=confidence,
            recommendation=analysis["recommendation"],
            possibleConditions=analysis["conditions"],
            analysis=f"Based on the symptoms described, the AI model has identified potential conditions. Primary match: {analysis.get('matched_keyword', 'general symptoms')}.",
            success=True
        )
    except Exception as e:
        logger.error(f"Error analyzing symptoms: {str(e)}")
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=f"Error analyzing symptoms: {str(e)}")

@app.get("/ai/symptom-check/conditions")
async def get_supported_conditions(api_key: str = Depends(verify_api_key)):
    return {"conditions": list(CONDITIONS_DB.keys()), "total": len(CONDITIONS_DB)}

# --- Image Diagnosis Routes ---
@app.post("/ai/image-diagnose", response_model=ImageDiagnosisResponse)
async def diagnose_image(
    file: UploadFile = File(...),
    x_image_type: str = Header(None),
    api_key: str = Depends(verify_api_key)
):
    try:
        if file.content_type not in settings.SUPPORTED_IMAGE_TYPES:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail=f"Unsupported file type. Supported types: {settings.SUPPORTED_IMAGE_TYPES}")
        
        image_bytes = await file.read()
        if len(image_bytes) > settings.MAX_IMAGE_SIZE:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail=f"File too large. Max size: {settings.MAX_IMAGE_SIZE / (1024*1024)}MB")
        
        logger.info(f"Analyzing image: {file.filename}, Size: {len(image_bytes)} bytes")
        image = preprocess_image(image_bytes)
        analysis = analyze_image_cnn(image)
        condition = analysis["primary_condition"]
        
        return ImageDiagnosisResponse(
            predictedCondition=condition["name"],
            confidenceScore=analysis["confidence"],
            possibleConditions=analysis["alternatives"],
            recommendation=condition["recommendation"],
            analysis=f"AI analysis detected {condition['name']}. {condition['description']}",
            success=True
        )
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"Error analyzing image: {str(e)}")
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=f"Error analyzing image: {str(e)}")

@app.get("/ai/image-diagnose/conditions")
async def get_image_conditions(api_key: str = Depends(verify_api_key)):
    return {
        "conditions": [{"id": k, "name": v["name"], "risk_level": v["risk"]} for k, v in SKIN_CONDITIONS.items()],
        "total": len(SKIN_CONDITIONS)
    }

# --- OCR Routes ---
@app.post("/ai/extract-text", response_model=OCRResponse)
async def extract_text(
    file: UploadFile = File(...),
    x_file_type: str = Header(None),
    api_key: str = Depends(verify_api_key)
):
    try:
        supported_types = settings.SUPPORTED_IMAGE_TYPES + ['application/pdf']
        if file.content_type not in supported_types:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail=f"Unsupported file type. Supported types: {supported_types}")
        
        file_bytes = await file.read()
        if len(file_bytes) > settings.MAX_IMAGE_SIZE:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail=f"File too large. Max size: {settings.MAX_IMAGE_SIZE / (1024*1024)}MB")
        
        logger.info(f"Processing OCR for: {file.filename}")
        
        if file.content_type == 'application/pdf':
            extracted_text, confidence = mock_ocr_extraction(file.filename)
        else:
            image = Image.open(io.BytesIO(file_bytes))
            if image.mode != 'RGB':
                image = image.convert('RGB')
            extracted_text, confidence = mock_ocr_extraction(file.filename)
        
        analysis = analyze_medical_text(extracted_text)
        
        return OCRResponse(
            extractedText=extracted_text,
            confidence=round(confidence, 2),
            analysis=analysis,
            success=True
        )
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"Error in OCR: {str(e)}")
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=f"Error extracting text: {str(e)}")

@app.get("/ai/ocr/status")
async def get_ocr_status(api_key: str = Depends(verify_api_key)):
    return {
        "tesseract_available": TESSERACT_AVAILABLE,
        "mock_mode": settings.ENABLE_MOCK_MODELS,
        "supported_formats": settings.SUPPORTED_IMAGE_TYPES + ['application/pdf']
    }

# --- Risk Prediction Routes ---
@app.post("/ai/risk-predict", response_model=RiskPredictionResponse)
async def predict_risk(request: RiskPredictionRequest, api_key: str = Depends(verify_api_key)):
    try:
        logger.info(f"Predicting risk for age {request.age}")
        
        if request.systolic < 70 or request.systolic > 250:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Systolic BP should be between 70 and 250")
        if request.diastolic < 40 or request.diastolic > 150:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Diastolic BP should be between 40 and 150")
        if request.bloodSugar < 50 or request.bloodSugar > 600:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Blood sugar should be between 50 and 600")
        if request.heartRate < 30 or request.heartRate > 200:
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Heart rate should be between 30 and 200")
        
        prediction = predict_health_risk_ml(request)
        
        return RiskPredictionResponse(
            riskScore=prediction["risk_score"],
            riskCategory=prediction["risk_category"],
            factors=prediction["factors"],
            recommendation=prediction["recommendation"],
            success=True
        )
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"Error predicting risk: {str(e)}")
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=f"Error predicting risk: {str(e)}")

@app.get("/ai/risk-predict/reference-ranges")
async def get_reference_ranges(api_key: str = Depends(verify_api_key)):
    return {
        "blood_pressure": {
            "normal": {"systolic": "< 120", "diastolic": "< 80"},
            "elevated": {"systolic": "120-129", "diastolic": "< 80"},
            "stage1": {"systolic": "130-139", "diastolic": "80-89"},
            "stage2": {"systolic": ">= 140", "diastolic": ">= 90"},
            "crisis": {"systolic": ">= 180", "diastolic": ">= 120"}
        },
        "blood_sugar": {
            "normal": "< 100 mg/dL (fasting)",
            "pre_diabetic": "100-125 mg/dL",
            "diabetic": ">= 126 mg/dL"
        },
        "heart_rate": {
            "normal": "60-100 BPM",
            "bradycardia": "< 60 BPM",
            "tachycardia": "> 100 BPM"
        }
    }

# Exception handler
@app.exception_handler(Exception)
async def global_exception_handler(request, exc):
    logger.error(f"Unhandled exception: {str(exc)}")
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={"success": False, "error": "Internal server error", "message": str(exc)}
    )

print("✅ FastAPI app created successfully!")

## Step 4: Start the Server with Ngrok

In [ ]:
import threading
import uvicorn
import time

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start server in background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

print("✅ Server started on http://localhost:8000")

# Create ngrok tunnel for public access
try:
    public_url = ngrok.connect(8000)
    print(f"\n🌐 Public URL: {public_url}")
    print(f"\n📋 API Documentation: {public_url}/docs")
    print(f"🔧 Alternative docs: {public_url}/redoc")
    print("\n⚠️ Note: The API key is: ai-service-secret-key")
    print("   Include 'X-API-Key: ai-service-secret-key' header in all requests")
except Exception as e:
    print(f"⚠️ Could not create ngrok tunnel: {e}")
    print("Server is still running locally at http://localhost:8000")

## Step 5: Test the API

In [ ]:
import httpx
import json

# Test the health endpoint
print("Testing API Health...")
response = httpx.get("http://localhost:8000/health")
print(f"Health Check: {response.json()}")

# Test symptom checker
print("\nTesting Symptom Checker...")
headers = {"X-API-Key": "ai-service-secret-key"}
symptom_data = {
    "symptoms": "I have fever and headache for 2 days",
    "age": 30,
    "gender": "male"
}
response = httpx.post("http://localhost:8000/ai/symptom-check", json=symptom_data, headers=headers)
print(f"Symptom Check Result: {json.dumps(response.json(), indent=2)}")

# Test risk prediction
print("\nTesting Risk Prediction...")
risk_data = {
    "age": 45,
    "gender": "male",
    "systolic": 135,
    "diastolic": 85,
    "bloodSugar": 110,
    "heartRate": 78
}
response = httpx.post("http://localhost:8000/ai/risk-predict", json=risk_data, headers=headers)
print(f"Risk Prediction Result: {json.dumps(response.json(), indent=2)}")

## API Endpoints Summary

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/` | GET | Root endpoint |
| `/health` | GET | Health check |
| `/ai/symptom-check` | POST | Analyze symptoms |
| `/ai/symptom-check/conditions` | GET | Get supported conditions |
| `/ai/image-diagnose` | POST | Analyze medical image |
| `/ai/image-diagnose/conditions` | GET | Get skin conditions |
| `/ai/extract-text` | POST | OCR text extraction |
| `/ai/ocr/status` | GET | OCR service status |
| `/ai/risk-predict` | POST | Predict health risk |
| `/ai/risk-predict/reference-ranges` | GET | Get reference ranges |

**All `/ai/*` endpoints require `X-API-Key` header with value `ai-service-secret-key`**

## Step 6: Keep Server Running

Run this cell to keep the server alive. The server will continue running until you stop the notebook runtime.

In [ ]:
# Keep the server running
print("Server is running... Press the Stop button (■) to stop.")
print(f"\n🌐 Public URL: {public_url if 'public_url' in dir() else 'Not available'}")
print(f"📋 API Docs: {public_url}/docs" if 'public_url' in dir() else "Local: http://localhost:8000/docs")

# This will keep the cell running indefinitely
import time
while True:
    time.sleep(60)

## Configuration for Spring Boot Backend

Update your `application.yml` in the Spring Boot backend:

```yaml
ai:
  service:
    url: YOUR_NGROK_URL  # e.g., https://abc123.ngrok.io
    api-key: ai-service-secret-key
```

Replace `YOUR_NGROK_URL` with the public URL printed above.